In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/nova-tf-idf/cm_tfidf.png
/kaggle/input/datasets/raesiow/nova-tf-idf/baseline_preds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv
/kaggle/input/datasets/raesiow/nova-tf-idf/cm_text_vs_both.png


In [2]:
!pip install -q sentence-transformers  # install the MiniLM library (-q = quiet output)

from sentence_transformers import SentenceTransformer  # loads MiniLM

nova = pd.read_csv("/kaggle/input/datasets/raesiow/nova-tf-idf/fpx_off_folds.csv", dtype={"off_code": str})  # the 1,884 NOVA rows with folds
print(nova.shape)  # expect (1884, 22)

minilm = SentenceTransformer("all-MiniLM-L6-v2")  # download the frozen pretrained MiniLM (needs Internet)
texts = nova["ingredients_text"].str.lower().tolist()  # ingredient texts as a plain list, lowercased
emb = minilm.encode(texts, batch_size=64, show_progress_bar=True)  # each text -> 384 numbers (no training, just reading)
print(emb.shape)  # expect (1884, 384): one row per product, 384 columns

np.save("/kaggle/working/minilm_nova_emb.npy", emb)  # save, so we don't have to encode again if the session resets

(1884, 22)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

(1884, 384)


In [3]:
import torch  # PyTorch for the MLP
from sklearn.preprocessing import StandardScaler  # scaling, fit on training folds only
from sklearn.utils.class_weight import compute_class_weight  # balanced class weights
from sklearn.metrics import f1_score  # macro-F1

nut_cols = ["protein_g_f", "fat_g_f", "carb_excl_fiber_g_f", "sugars_g_f", "fiber_g_f", "sat_fat_g_f", "sodium_g_f"]  # same 7 nutrients as before
nut = np.log1p(nova[nut_cols].values)  # log(1 + grams), same as the other NOVA models
x_all = np.hstack([emb, nut])  # glue side by side: 384 MiniLM numbers + 7 nutrients = 391 columns
y_all = nova["nova"].values - 1  # NOVA 1-4 -> classes 0-3 (PyTorch wants them from 0)
print(x_all.shape)  # expect (1884, 391)

def make_model(n_inputs):  # builds a fresh MLP; n_inputs lets us reuse it later for text-only (384)
    return torch.nn.Sequential(  # layers run in order, top to bottom
        torch.nn.Linear(n_inputs, 64),  # 391 inputs -> 64 hidden units
        torch.nn.ReLU(),  # negatives -> 0, so it can learn curves
        torch.nn.Dropout(0.3),  # switch off 30% of hidden units at random during training
        torch.nn.Linear(64, 4))  # 64 hidden units -> 4 scores, one per NOVA class

def train_one_fold(x_train, y_train, x_test):  # trains on one fold's training rows, returns probabilities for its test rows
    torch.manual_seed(42)  # same random start every time, so reruns match
    model = make_model(x_train.shape[1])  # fresh model, input size taken from the data
    w = compute_class_weight("balanced", classes=np.arange(4), y=y_train)  # rarer class -> bigger weight (NOVA 2!)
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32))  # classification loss with class weights
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # optimiser, same as the GI MLP
    xt = torch.tensor(x_train, dtype=torch.float32)  # training inputs as a tensor
    yt = torch.tensor(y_train, dtype=torch.long)  # training labels as whole numbers (CrossEntropy needs long)
    model.train()  # training mode: dropout ON
    for epoch in range(300):  # 300 passes over all training rows (full batch)
        opt.zero_grad()  # clear old gradients
        loss = loss_fn(model(xt), yt)  # forward pass + how wrong it is
        loss.backward()  # compute gradients
        opt.step()  # nudge the weights
    model.eval()  # prediction mode: dropout OFF
    with torch.no_grad():  # no gradients needed to predict
        logits = model(torch.tensor(x_test, dtype=torch.float32))  # 4 scores per test row
    return torch.softmax(logits, dim=1).numpy()  # scores -> probabilities that add to 1

(1884, 391)


In [4]:
probs_all = np.zeros((len(nova), 4))  # out-of-fold probabilities: one row per product, 4 columns
scores = []  # macro-F1 per fold

for k in range(5):  # each fold takes a turn as test
    tr = (nova["fold"] != k).values  # True for training rows
    te = ~tr  # True for test rows (~ flips True/False)
    scaler = StandardScaler().fit(x_all[tr])  # learn mean/spread from training rows only
    p = train_one_fold(scaler.transform(x_all[tr]), y_all[tr], scaler.transform(x_all[te]))  # train, predict test fold
    probs_all[te] = p  # store this fold's probabilities in the right rows
    f1 = f1_score(y_all[te], p.argmax(axis=1), average="macro")  # predicted class = biggest probability
    scores.append(f1)  # keep the score
    print(f"fold {k}: macro-F1 = {f1:.3f}")  # show each fold

print(f"MiniLM + nutrients MLP: mean = {np.mean(scores):.3f} +/- {np.std(scores):.3f}")  # compare: DistilBERT+nut GBM 0.784, LR 0.695

fold 0: macro-F1 = 0.733
fold 1: macro-F1 = 0.699
fold 2: macro-F1 = 0.733
fold 3: macro-F1 = 0.690
fold 4: macro-F1 = 0.590
MiniLM + nutrients MLP: mean = 0.689 +/- 0.052


In [5]:
scores_text = []  # macro-F1 per fold, text only

for k in range(5):  # same 5 folds
    tr = (nova["fold"] != k).values  # training rows
    te = ~tr  # test rows
    scaler = StandardScaler().fit(emb[tr])  # scale using training rows only (emb = 384 MiniLM columns, no nutrients)
    p = train_one_fold(scaler.transform(emb[tr]), y_all[tr], scaler.transform(emb[te]))  # same function; make_model sees 384 inputs
    f1 = f1_score(y_all[te], p.argmax(axis=1), average="macro")  # macro-F1 on the test fold
    scores_text.append(f1)  # keep it
    print(f"fold {k}: macro-F1 = {f1:.3f}")  # show each fold

print(f"MiniLM text-only MLP: mean = {np.mean(scores_text):.3f} +/- {np.std(scores_text):.3f}")  # compare: DistilBERT text-only 0.680

fold 0: macro-F1 = 0.714
fold 1: macro-F1 = 0.700
fold 2: macro-F1 = 0.742
fold 3: macro-F1 = 0.660
fold 4: macro-F1 = 0.579
MiniLM text-only MLP: mean = 0.679 +/- 0.057


In [6]:
out = nova[["fpx_code", "off_code", "nova", "fold"]].copy()  # IDs, true label, fold
out[["p1", "p2", "p3", "p4"]] = probs_all  # MiniLM + nutrients out-of-fold probabilities
out.to_csv("/kaggle/working/minilm_nut_oof.csv", index=False)  # save; download it before the session ends

In [7]:
def make_model(n_inputs, hidden=64, dropout=0.3):  # same MLP, but hidden size and dropout can now change (defaults = old settings)
    return torch.nn.Sequential(  # layers run in order
        torch.nn.Linear(n_inputs, hidden),  # inputs -> hidden units
        torch.nn.ReLU(),  # negatives -> 0
        torch.nn.Dropout(dropout),  # switch off this share of hidden units during training
        torch.nn.Linear(hidden, 4))  # hidden units -> 4 NOVA scores

def train_one_fold(x_train, y_train, x_test, hidden=64, dropout=0.3):  # same training as before, plus the two settings
    torch.manual_seed(42)  # same random start every time
    model = make_model(x_train.shape[1], hidden, dropout)  # fresh model with the chosen settings
    w = compute_class_weight("balanced", classes=np.arange(4), y=y_train)  # balanced class weights
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32))  # weighted classification loss
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # same optimiser as before
    xt = torch.tensor(x_train, dtype=torch.float32)  # inputs as tensor
    yt = torch.tensor(y_train, dtype=torch.long)  # labels as whole numbers
    model.train()  # dropout ON
    for epoch in range(300):  # 300 full-batch passes, same as before
        opt.zero_grad()  # clear old gradients
        loss = loss_fn(model(xt), yt)  # forward + loss
        loss.backward()  # gradients
        opt.step()  # update weights
    model.eval()  # dropout OFF
    with torch.no_grad():  # no gradients for prediction
        logits = model(torch.tensor(x_test, dtype=torch.float32))  # 4 scores per test row
    return torch.softmax(logits, dim=1).numpy()  # probabilities

def inner_score(train_rows, hidden, dropout):  # average validation macro-F1 of one setting, using ONLY the outer training rows
    fold_col = nova["fold"].values  # fold number of every row
    f1s = []  # validation scores
    for j in np.unique(fold_col[train_rows]):  # the 4 fold numbers inside the outer training part
        in_train = train_rows & (fold_col != j)  # inner training rows: outer training rows, minus fold j
        in_val = train_rows & (fold_col == j)  # inner validation rows: fold j
        scaler = StandardScaler().fit(x_all[in_train])  # scaling from inner training rows only
        p = train_one_fold(scaler.transform(x_all[in_train]), y_all[in_train], scaler.transform(x_all[in_val]), hidden, dropout)  # train + predict validation
        f1s.append(f1_score(y_all[in_val], p.argmax(axis=1), average="macro"))  # validation macro-F1
    return np.mean(f1s)  # average over the 4 inner turns

In [8]:
import time  # to time the run
settings = [(h, d) for h in [32, 64, 128] for d in [0.2, 0.3, 0.5]]  # the 9 combinations, fixed in advance
nested_scores = []  # outer test macro-F1 per fold (the honest score)
probs_tuned = np.zeros((len(nova), 4))  # out-of-fold probabilities from the tuned models
t0 = time.time()  # start timer

for k in range(5):  # outer loop: each fold takes a turn as test
    tr = (nova["fold"] != k).values  # outer training rows
    te = ~tr  # outer test rows (not touched while choosing)
    val_scores = {}  # setting -> average validation macro-F1
    for hidden, dropout in settings:  # try every combination
        val_scores[(hidden, dropout)] = inner_score(tr, hidden, dropout)  # inner turns on the 4 training folds only
    best_hidden, best_dropout = max(val_scores, key=val_scores.get)  # combination with the highest validation score
    scaler = StandardScaler().fit(x_all[tr])  # refit scaling on all 4 training folds
    p = train_one_fold(scaler.transform(x_all[tr]), y_all[tr], scaler.transform(x_all[te]), best_hidden, best_dropout)  # refit with best setting, predict test fold
    probs_tuned[te] = p  # store test probabilities
    f1 = f1_score(y_all[te], p.argmax(axis=1), average="macro")  # score on the untouched test fold
    nested_scores.append(f1)  # keep it
    print(f"fold {k}: best hidden={best_hidden}, dropout={best_dropout} (validation {val_scores[(best_hidden, best_dropout)]:.3f}) -> test macro-F1 = {f1:.3f}   ({time.time() - t0:.0f} s)")  # what was chosen, and how it did

print(f"MiniLM + nutrients MLP, nested-CV tuned: mean = {np.mean(nested_scores):.3f} +/- {np.std(nested_scores):.3f}")  # compare: untuned 0.689 +/- 0.052

fold 0: best hidden=32, dropout=0.2 (validation 0.676) -> test macro-F1 = 0.734   (49 s)
fold 1: best hidden=64, dropout=0.5 (validation 0.687) -> test macro-F1 = 0.709   (97 s)
fold 2: best hidden=32, dropout=0.5 (validation 0.666) -> test macro-F1 = 0.703   (144 s)
fold 3: best hidden=128, dropout=0.2 (validation 0.695) -> test macro-F1 = 0.684   (193 s)
fold 4: best hidden=64, dropout=0.5 (validation 0.706) -> test macro-F1 = 0.598   (241 s)
MiniLM + nutrients MLP, nested-CV tuned: mean = 0.685 +/- 0.047


In [9]:
def make_model(n_inputs, hidden=64, dropout=0.3, n_layers=1):  # n_layers = 1 or 2 hidden layers
    layers = [torch.nn.Linear(n_inputs, hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout)]  # first hidden layer, as before
    last_size = hidden  # size of the most recent layer
    if n_layers == 2:  # only when we want a second hidden layer
        layers += [torch.nn.Linear(hidden, hidden // 2), torch.nn.ReLU(), torch.nn.Dropout(dropout)]  # second layer, half the size (// = whole-number division)
        last_size = hidden // 2  # the output layer now starts from this smaller size
    layers.append(torch.nn.Linear(last_size, 4))  # output: 4 NOVA scores
    return torch.nn.Sequential(*layers)  # * unpacks the list into separate layers, in order

def train_one_fold(x_train, y_train, x_test, hidden=64, dropout=0.3, n_layers=1):  # same as before + n_layers
    torch.manual_seed(42)  # same random start
    model = make_model(x_train.shape[1], hidden, dropout, n_layers)  # pass all three settings on
    w = compute_class_weight("balanced", classes=np.arange(4), y=y_train)  # balanced class weights
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32))  # weighted loss
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # same optimiser
    xt = torch.tensor(x_train, dtype=torch.float32)  # inputs
    yt = torch.tensor(y_train, dtype=torch.long)  # labels
    model.train()  # dropout ON
    for epoch in range(300):  # 300 full-batch passes
        opt.zero_grad()  # clear gradients
        loss = loss_fn(model(xt), yt)  # forward + loss
        loss.backward()  # gradients
        opt.step()  # update
    model.eval()  # dropout OFF
    with torch.no_grad():  # no gradients
        logits = model(torch.tensor(x_test, dtype=torch.float32))  # scores
    return torch.softmax(logits, dim=1).numpy()  # probabilities

def inner_score(train_rows, hidden, dropout, n_layers):  # same as before + n_layers
    fold_col = nova["fold"].values  # fold number of every row
    f1s = []  # validation scores
    for j in np.unique(fold_col[train_rows]):  # the 4 inner folds
        in_train = train_rows & (fold_col != j)  # inner training rows
        in_val = train_rows & (fold_col == j)  # inner validation rows
        scaler = StandardScaler().fit(x_all[in_train])  # scaling from inner training only
        p = train_one_fold(scaler.transform(x_all[in_train]), y_all[in_train], scaler.transform(x_all[in_val]), hidden, dropout, n_layers)  # train + predict
        f1s.append(f1_score(y_all[in_val], p.argmax(axis=1), average="macro"))  # validation macro-F1
    return np.mean(f1s)  # average of 4 inner turns

In [10]:
settings = [(n, h, d) for n in [1, 2] for h in [32, 64, 128] for d in [0.2, 0.3, 0.5]]  # 2 x 3 x 3 = 18 combinations, fixed in advance
nested_scores2 = []  # outer test macro-F1 per fold
t0 = time.time()  # start timer

for k in range(5):  # outer loop: each fold is test once
    tr = (nova["fold"] != k).values  # outer training rows
    te = ~tr  # outer test rows, untouched while choosing
    val_scores = {}  # setting -> validation macro-F1
    for n_layers, hidden, dropout in settings:  # try all 18
        val_scores[(n_layers, hidden, dropout)] = inner_score(tr, hidden, dropout, n_layers)  # inner turns only
    best_n, best_h, best_d = max(val_scores, key=val_scores.get)  # best combination by validation
    scaler = StandardScaler().fit(x_all[tr])  # refit scaling on all 4 training folds
    p = train_one_fold(scaler.transform(x_all[tr]), y_all[tr], scaler.transform(x_all[te]), best_h, best_d, best_n)  # refit with best, predict test
    f1 = f1_score(y_all[te], p.argmax(axis=1), average="macro")  # score on untouched test fold
    nested_scores2.append(f1)  # keep it
    print(f"fold {k}: best layers={best_n}, hidden={best_h}, dropout={best_d} (validation {val_scores[(best_n, best_h, best_d)]:.3f}) -> test macro-F1 = {f1:.3f}   ({time.time() - t0:.0f} s)")  # what was chosen

print(f"nested-CV, 1 or 2 layers: mean = {np.mean(nested_scores2):.3f} +/- {np.std(nested_scores2):.3f}")  # compare: 0.685 (1 layer tuned), 0.689 (untuned)

fold 0: best layers=1, hidden=32, dropout=0.2 (validation 0.676) -> test macro-F1 = 0.734   (109 s)
fold 1: best layers=1, hidden=64, dropout=0.5 (validation 0.687) -> test macro-F1 = 0.709   (218 s)
fold 2: best layers=2, hidden=32, dropout=0.5 (validation 0.671) -> test macro-F1 = 0.687   (326 s)
fold 3: best layers=1, hidden=128, dropout=0.2 (validation 0.695) -> test macro-F1 = 0.684   (437 s)
fold 4: best layers=1, hidden=64, dropout=0.5 (validation 0.706) -> test macro-F1 = 0.598   (546 s)
nested-CV, 1 or 2 layers: mean = 0.682 +/- 0.046


In [11]:
nova["text_len"] = nova["ingredients_text"].str.len()  # number of characters in each ingredient text
print(nova["text_len"].describe())  # min, median, max length

print("=== 5 LONGEST ===")  # header
for t in nova.sort_values("text_len", ascending=False)["ingredients_text"].head(5):  # 5 longest texts
    print(len(t), "|", t[:300], "\n")  # length, then first 300 characters (enough to judge if it's clean)

print("=== 5 SHORTEST ===")  # header
for t in nova.sort_values("text_len")["ingredients_text"].head(5):  # 5 shortest texts
    print(len(t), "|", t, "\n")  # length and the whole text

nova["correct"] = probs_all.argmax(axis=1) == y_all  # True if the MiniLM + nutrients model got this row right (out-of-fold)
nova["length_group"] = pd.qcut(nova["text_len"], 5)  # split into 5 equal-size groups, shortest to longest
print(nova.groupby("length_group", observed=True)["correct"].mean().round(3))  # share correct in each length group

count    1884.000000
mean      216.786624
std       230.046519
min         3.000000
25%        41.000000
50%       149.000000
75%       324.000000
max      2563.000000
Name: text_len, dtype: float64
=== 5 LONGEST ===
2563 | Salted Napa Cabbage 69.1% (Napa Cabbage, Salt, Radish, Red Pepper Seasoned Paste 10% (Garlic, Sweetener (Sorbitol, Water, Red Pepper Powder, Radish, Onion, Flavour Enhancer (Monosodium Glutamate), Salt, Ginger, Thickener (Xanthan Gumil, Green Onion, Thickening Agent (Water, Thickener (Xanthan Gum, S 

1675 | Econstituted Skimmed Milk, Polydextrose (Dietary Fibre), Sugar, Cookie Dough ieces (6.1%) (Sugar, Wheat Flour, Butter (Milk), Glucose-Fructose Syrup, Water, Salt, Flavouring (Barley), Vanilla Powder), Cream (Milk), Sweetener: Erythritol; Eggs, Skimmed Milk Powder, Chocolate Drops (1.9 %) (Sugar, Coc 

1574 | Rice chex : whole grain rice, rice, sugar, salt, molasses. vitamin e (mixed tocopherols) added to preserve freshness.vitamins and minerals: calcium carbonat

In [12]:
group_30_95 = nova[(nova["text_len"] > 30) & (nova["text_len"] <= 95) & (~nova["correct"])].copy()  # wrong predictions in the 30-95 character group
group_30_95["pred_nova"] = probs_all[group_30_95.index].argmax(axis=1) + 1  # the model's guess, as NOVA 1-4
print(len(group_30_95), "wrong in this group")  # how many mistakes
print(group_30_95[["fpx_name", "ingredients_text", "nova", "pred_nova"]].sample(10, random_state=1).to_string())  # 10 random mistakes: FoodProX name, text, true label, guess

120 wrong in this group
                                                   fpx_name                                                                                ingredients_text  nova  pred_nova
1588  tangerines mandarin oranges canned juice pack drained                     Mandarins, mandarin juice from concentrate, acidity regulator: citric acid.     3          1
960                                 snacks brown rice chips                                   Jasmine Brown Rice 94.5%, Water, Sea Salt (Certified Organic)     4          1
1028                                              apple raw                                   Apples, Maple flavor, Caramel color, Lemon juice concentrate.     1          3
1871                         grapefruit raw white all areas                 Grapefruit, Grapefruit Juice From Concentrate, Firming Agent (Calcium Chloride)     1          3
1361                     babyfood fruit applesauce strained                                       Organic apple

Of 10 randomly inspected errors among 30–95-character texts, about half were matching mismatches where the OFF product was more processed than the generic FoodProX food (e.g. ‘apple, raw’ matched to a product with caramel colour; ‘herring’ matched to salted salmon), and in several of these the model’s prediction fitted the product better than the label. The measured macro-F1 therefore likely underestimates true performance, and improving matching (e.g. a processing-aware check) is the main route to better results, rather than model tuning.